# Metric Presentation and Visualization
## Necessary packages and functions call

- DDPM-TS: Interpretable Diffusion for Time Series Generation
- Metrics: 
    - discriminative_metrics
    - predictive_metrics
    - visualization

In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import sys
sys.path.append(os.path.join(os.path.dirname('__file__'), '../'))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import tensorflow as tf
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

from Utils.metric_utils import display_scores
from Utils.discriminative_metric import discriminative_score_metrics
from Utils.predictive_metric import predictive_score_metrics

## Data Loading ETTh Morning Peak

Load original dataset and preprocess the loaded data.

In [3]:
# iterations = 5
# ori_data = np.load('../toy_exp/samples/sine_ground_truth_24_train.npy')
# # ori_data = np.load('../OUTPUT/{dataset_name}/samples/{dataset_name}_norm_truth_{seq_length}_train.npy')  # Uncomment the line if dataset other than Sine is used.
# fake_data = np.load('../toy_exp/ddpm_fake_sines.npy')


iterations = 5
# ori_data = np.load('../toy_exp/samples/sine_ground_truth_24_train.npy')
# ori_data = np.load('../OUTPUT/test_ep/samples/etth_norm_truth_24_train.npy')  # Uncomment the line if dataset other than Sine is used.
# fake_data = np.load('../OUTPUT/test_ep/ddpm_fake_test_ep_milestone_10.npy')

ori_data = np.load('../OUTPUT/etthmpep_energympep/samples/morning_peak_etth_norm_truth_24_train.npy')  # Uncomment the line if dataset other than Sine is used.
fake_data = np.load('../OUTPUT/etthmpep_energympep/ddpm_fake_morning_peak_etth_milestone_500_mask0.0.npy')

print('ori shape is: ', ori_data.shape)
print('fake shape is: ', fake_data.shape)
b,t,n = ori_data.shape


ori_data = ori_data.transpose(2, 0, 1).reshape(b * n, t, 1)

# fake_data = fake_data[:ori_data.shape[0]*ori_data.shape[2]]
# fake_data = fake_data.reshape(n, b, t).transpose(1, 2, 0)

print('ori shape is: ', ori_data.shape)
print('fake shape is: ', fake_data.shape)

ori shape is:  (2881, 24, 7)
fake shape is:  (20700, 24, 1)
ori shape is:  (20167, 24, 1)
fake shape is:  (20700, 24, 1)


## Evaluate the generated data

### 1. Discriminative score

To evaluate the classification accuracy between original and synthetic data using post-hoc RNN network. The output is | classification accuracy - 0.5 |.

- metric_iteration: the number of iterations for metric computation.

In [4]:
discriminative_score = []

for i in range(iterations):
    temp_disc, fake_acc, real_acc = discriminative_score_metrics(ori_data[:], fake_data[:ori_data.shape[0]])
    discriminative_score.append(temp_disc)
    print(f'Iter {i}: ', temp_disc, ',', fake_acc, ',', real_acc, '\n')
      
print('etth:')
display_scores(discriminative_score)
print()
# seed 12345  Final Score:  0.0731896551724138 ± 0.005795392020461213
# seed 2024  Final Score:  0.06945402298850574 ± 0.010002021083688875

# univariate Final Score:  0.004115022310361938 ± 0.003740431320553614


Instructions for updating:
Please use `keras.layers.RNN(cell)`, which is equivalent to this API
Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor
Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor
Instructions for updating:
Please use tf.global_variables instead.


training: 100%|██████████| 2000/2000 [00:43<00:00, 45.76it/s]


Iter 0:  0.01152702032721864 , 0.5443728309370351 , 0.4786812097174021 



training: 100%|██████████| 2000/2000 [00:44<00:00, 45.20it/s]


Iter 1:  0.010039662865642063 , 0.5639563708477937 , 0.4561229548834903 



...

Iter 2:  0.009295984134853719 , 0.4816559246405553 , 0.4997521070897372 



training: 100%|██████████| 2000/2000 [00:43<00:00, 45.70it/s]


Iter 3:  0.004586018839861206 , 0.5052057511155181 , 0.5039662865642043 



training: 100%|██████████| 2000/2000 [00:43<00:00, 45.66it/s]


Iter 4:  0.01313832424392658 , 0.5267724343083788 , 0.4995042141794745 

etth:
Final Score:  0.00971740208230044 ± 0.0040037631170122285



## Evaluate the generated data

### 2. Predictive score

To evaluate the prediction performance on train on synthetic, test on real setting. More specifically, we use Post-hoc RNN architecture to predict one-step ahead and report the performance in terms of MAE. 

The model learns to predict the last dimension with one more step.

In [5]:
predictive_score = []
for i in range(iterations):
    temp_pred = predictive_score_metrics(ori_data, fake_data[:ori_data.shape[0]])
    predictive_score.append(temp_pred)
    print(i, ' epoch: ', temp_pred, '\n')
      
print('sine:')
display_scores(predictive_score)
print()

# univariate Final Score:  0.1605687830457955 ± 3.0256014600636085e-05

training: 100%|██████████| 5000/5000 [01:45<00:00, 47.19it/s]


0  epoch:  0.1606272078960458 



training: 100%|██████████| 5000/5000 [01:47<00:00, 46.56it/s]


1  epoch:  0.1605913799635576 



training: 100%|██████████| 5000/5000 [01:44<00:00, 48.05it/s]


2  epoch:  0.1607455182582338 



training: 100%|██████████| 5000/5000 [01:41<00:00, 49.18it/s]


3  epoch:  0.16080090367693337 



training: 100%|██████████| 5000/5000 [01:40<00:00, 49.98it/s]


4  epoch:  0.16061289231885376 

sine:
Final Score:  0.16067558042272487 ± 0.00011440273901185377



## Data Loading ETTh Evening Peak

Load original dataset and preprocess the loaded data.

In [6]:
# iterations = 5
# ori_data = np.load('../toy_exp/samples/sine_ground_truth_24_train.npy')
# # ori_data = np.load('../OUTPUT/{dataset_name}/samples/{dataset_name}_norm_truth_{seq_length}_train.npy')  # Uncomment the line if dataset other than Sine is used.
# fake_data = np.load('../toy_exp/ddpm_fake_sines.npy')


iterations = 5
# ori_data = np.load('../toy_exp/samples/sine_ground_truth_24_train.npy')
# ori_data = np.load('../OUTPUT/test_ep/samples/etth_norm_truth_24_train.npy')  # Uncomment the line if dataset other than Sine is used.
# fake_data = np.load('../OUTPUT/test_ep/ddpm_fake_test_ep_milestone_10.npy')

ori_data = np.load('../OUTPUT/etthmpep_energympep/samples/evening_peak_etth_norm_truth_24_train.npy')  # Uncomment the line if dataset other than Sine is used.
fake_data = np.load('../OUTPUT/etthmpep_energympep/ddpm_fake_evening_peak_etth_milestone_500_mask0.0.npy')

print('ori shape is: ', ori_data.shape)
print('fake shape is: ', fake_data.shape)
b,t,n = ori_data.shape


ori_data = ori_data.transpose(2, 0, 1).reshape(b * n, t, 1)

# fake_data = fake_data[:ori_data.shape[0]*ori_data.shape[2]]
# fake_data = fake_data.reshape(n, b, t).transpose(1, 2, 0)

print('ori shape is: ', ori_data.shape)
print('fake shape is: ', fake_data.shape)

ori shape is:  (2880, 24, 7)
fake shape is:  (20700, 24, 1)
ori shape is:  (20160, 24, 1)
fake shape is:  (20700, 24, 1)


## Evaluate the generated data

### 1. Discriminative score

To evaluate the classification accuracy between original and synthetic data using post-hoc RNN network. The output is | classification accuracy - 0.5 |.

- metric_iteration: the number of iterations for metric computation.

In [7]:
discriminative_score = []

for i in range(iterations):
    temp_disc, fake_acc, real_acc = discriminative_score_metrics(ori_data[:], fake_data[:ori_data.shape[0]])
    discriminative_score.append(temp_disc)
    print(f'Iter {i}: ', temp_disc, ',', fake_acc, ',', real_acc, '\n')
      
print('etth:')
display_scores(discriminative_score)
print()
# seed 12345  Final Score:  0.0731896551724138 ± 0.005795392020461213
# seed 2024  Final Score:  0.06945402298850574 ± 0.010002021083688875

# univariate Final Score:  0.004115022310361938 ± 0.003740431320553614


training: 100%|██████████| 2000/2000 [00:45<00:00, 43.57it/s]


Iter 0:  0.005332341269841279 , 0.5766369047619048 , 0.4340277777777778 



training: 100%|██████████| 2000/2000 [00:46<00:00, 43.05it/s]


Iter 1:  0.00868055555555558 , 0.47346230158730157 , 0.5438988095238095 



training: 100%|██████████| 2000/2000 [00:49<00:00, 40.70it/s]


Iter 2:  0.007192460317460347 , 0.5419146825396826 , 0.4724702380952381 



training: 100%|██████████| 2000/2000 [00:47<00:00, 42.28it/s]


Iter 3:  0.017857142857142905 , 0.5920138888888888 , 0.4437003968253968 



training: 100%|██████████| 2000/2000 [00:48<00:00, 40.89it/s]


Iter 4:  0.01810515873015872 , 0.5736607142857143 , 0.4625496031746032 

etth:
Final Score:  0.011433531746031766 ± 0.007567111267638556



## Evaluate the generated data

### 2. Predictive score

To evaluate the prediction performance on train on synthetic, test on real setting. More specifically, we use Post-hoc RNN architecture to predict one-step ahead and report the performance in terms of MAE. 

The model learns to predict the last dimension with one more step.

In [8]:
predictive_score = []
for i in range(iterations):
    temp_pred = predictive_score_metrics(ori_data, fake_data[:ori_data.shape[0]])
    predictive_score.append(temp_pred)
    print(i, ' epoch: ', temp_pred, '\n')
      
print('sine:')
display_scores(predictive_score)
print()

# univariate Final Score:  0.1605687830457955 ± 3.0256014600636085e-05

training: 100%|██████████| 5000/5000 [01:39<00:00, 50.50it/s]


0  epoch:  0.13087355065581097 



training: 100%|██████████| 5000/5000 [01:34<00:00, 52.70it/s]


1  epoch:  0.1308720244954108 



training: 100%|██████████| 5000/5000 [01:40<00:00, 49.59it/s]


2  epoch:  0.13091880259185032 



training: 100%|██████████| 5000/5000 [01:35<00:00, 52.09it/s]


3  epoch:  0.1309099433656711 



training: 100%|██████████| 5000/5000 [01:32<00:00, 54.29it/s]


4  epoch:  0.13089736979167152 

sine:
Final Score:  0.13089433818008295 ± 2.6202064430470004e-05



## Data Loading Energy Morning Peak

Load original dataset and preprocess the loaded data.

In [2]:
# iterations = 5
# ori_data = np.load('../toy_exp/samples/sine_ground_truth_24_train.npy')
# # ori_data = np.load('../OUTPUT/{dataset_name}/samples/{dataset_name}_norm_truth_{seq_length}_train.npy')  # Uncomment the line if dataset other than Sine is used.
# fake_data = np.load('../toy_exp/ddpm_fake_sines.npy')


iterations = 5
# ori_data = np.load('../toy_exp/samples/sine_ground_truth_24_train.npy')
# ori_data = np.load('../OUTPUT/test_ep/samples/etth_norm_truth_24_train.npy')  # Uncomment the line if dataset other than Sine is used.
# fake_data = np.load('../OUTPUT/test_ep/ddpm_fake_test_ep_milestone_10.npy')

ori_data = np.load('../OUTPUT/etthmpep_energympep/samples/morning_peak_energy_norm_truth_24_train.npy')  # Uncomment the line if dataset other than Sine is used.
fake_data = np.load('../OUTPUT/etthmpep_energympep/ddpm_fake_morning_peak_energy_milestone_500_mask0.0.npy')

print('ori shape is: ', ori_data.shape)
print('fake shape is: ', fake_data.shape)
b,t,n = ori_data.shape


ori_data = ori_data.transpose(2, 0, 1).reshape(b * n, t, 1)

# fake_data = fake_data[:ori_data.shape[0]*ori_data.shape[2]]
# fake_data = fake_data.reshape(n, b, t).transpose(1, 2, 0)

print('ori shape is: ', ori_data.shape)
print('fake shape is: ', fake_data.shape)

ori shape is:  (2580, 24, 28)
fake shape is:  (72900, 24, 1)
ori shape is:  (72240, 24, 1)
fake shape is:  (72900, 24, 1)


## Evaluate the generated data

### 1. Discriminative score

To evaluate the classification accuracy between original and synthetic data using post-hoc RNN network. The output is | classification accuracy - 0.5 |.

- metric_iteration: the number of iterations for metric computation.

In [3]:
discriminative_score = []

for i in range(iterations):
    temp_disc, fake_acc, real_acc = discriminative_score_metrics(ori_data[:], fake_data[:ori_data.shape[0]])
    discriminative_score.append(temp_disc)
    print(f'Iter {i}: ', temp_disc, ',', fake_acc, ',', real_acc, '\n')
      
print('etth:')
display_scores(discriminative_score)
print()
# seed 12345  Final Score:  0.0731896551724138 ± 0.005795392020461213
# seed 2024  Final Score:  0.06945402298850574 ± 0.010002021083688875

# univariate Final Score:  0.004115022310361938 ± 0.003740431320553614


Instructions for updating:
Please use `keras.layers.RNN(cell)`, which is equivalent to this API
Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor
Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor
Instructions for updating:
Please use tf.global_variables instead.


training: 100%|██████████| 2000/2000 [00:59<00:00, 33.37it/s]


Iter 0:  0.016196013289036526 , 0.1296373200442968 , 0.9027547065337763 



training: 100%|██████████| 2000/2000 [01:00<00:00, 33.10it/s]


Iter 1:  0.011558693244739793 , 0.48795681063122925 , 0.5351605758582503 



...

Iter 2:  0.013738925802879276 , 0.5047757475083057 , 0.5227021040974529 



training: 100%|██████████| 2000/2000 [00:54<00:00, 36.90it/s]


Iter 3:  0.008790143964562569 , 0.49750830564784054 , 0.5200719822812846 



training: 100%|██████████| 2000/2000 [00:54<00:00, 36.49it/s]


Iter 4:  0.0026993355481727543 , 0.47619047619047616 , 0.5184108527131783 

etth:
Final Score:  0.010596622369878184 ± 0.006445400352890955



## Evaluate the generated data

### 2. Predictive score

To evaluate the prediction performance on train on synthetic, test on real setting. More specifically, we use Post-hoc RNN architecture to predict one-step ahead and report the performance in terms of MAE. 

The model learns to predict the last dimension with one more step.

In [4]:
predictive_score = []
for i in range(iterations):
    temp_pred = predictive_score_metrics(ori_data, fake_data[:ori_data.shape[0]])
    predictive_score.append(temp_pred)
    print(i, ' epoch: ', temp_pred, '\n')
      
print('sine:')
display_scores(predictive_score)
print()

# univariate Final Score:  0.1605687830457955 ± 3.0256014600636085e-05

training: 100%|██████████| 5000/5000 [01:53<00:00, 43.88it/s]


0  epoch:  0.203633841147533 



training: 100%|██████████| 5000/5000 [01:49<00:00, 45.73it/s]


1  epoch:  0.20358360049044866 



training: 100%|██████████| 5000/5000 [01:54<00:00, 43.80it/s]


2  epoch:  0.2035897917578035 



training: 100%|██████████| 5000/5000 [01:58<00:00, 42.33it/s]


3  epoch:  0.20360067849684693 



training: 100%|██████████| 5000/5000 [01:57<00:00, 42.64it/s]


4  epoch:  0.20367051479574888 

sine:
Final Score:  0.2036156853376762 ± 4.503075169976743e-05



## Data Loading energy Evening Peak

Load original dataset and preprocess the loaded data.

In [5]:
# iterations = 5
# ori_data = np.load('../toy_exp/samples/sine_ground_truth_24_train.npy')
# # ori_data = np.load('../OUTPUT/{dataset_name}/samples/{dataset_name}_norm_truth_{seq_length}_train.npy')  # Uncomment the line if dataset other than Sine is used.
# fake_data = np.load('../toy_exp/ddpm_fake_sines.npy')


iterations = 5
# ori_data = np.load('../toy_exp/samples/sine_ground_truth_24_train.npy')
# ori_data = np.load('../OUTPUT/test_ep/samples/etth_norm_truth_24_train.npy')  # Uncomment the line if dataset other than Sine is used.
# fake_data = np.load('../OUTPUT/test_ep/ddpm_fake_test_ep_milestone_10.npy')

ori_data = np.load('../OUTPUT/etthmpep_energympep/samples/evening_peak_energy_norm_truth_24_train.npy')  # Uncomment the line if dataset other than Sine is used.
fake_data = np.load('../OUTPUT/etthmpep_energympep/ddpm_fake_evening_peak_energy_milestone_500_mask0.0.npy')

print('ori shape is: ', ori_data.shape)
print('fake shape is: ', fake_data.shape)
b,t,n = ori_data.shape


ori_data = ori_data.transpose(2, 0, 1).reshape(b * n, t, 1)

# fake_data = fake_data[:ori_data.shape[0]*ori_data.shape[2]]
# fake_data = fake_data.reshape(n, b, t).transpose(1, 2, 0)

print('ori shape is: ', ori_data.shape)
print('fake shape is: ', fake_data.shape)

ori shape is:  (2587, 24, 28)
fake shape is:  (72900, 24, 1)
ori shape is:  (72436, 24, 1)
fake shape is:  (72900, 24, 1)


## Evaluate the generated data

### 1. Discriminative score

To evaluate the classification accuracy between original and synthetic data using post-hoc RNN network. The output is | classification accuracy - 0.5 |.

- metric_iteration: the number of iterations for metric computation.

In [6]:
discriminative_score = []

for i in range(iterations):
    temp_disc, fake_acc, real_acc = discriminative_score_metrics(ori_data[:], fake_data[:ori_data.shape[0]])
    discriminative_score.append(temp_disc)
    print(f'Iter {i}: ', temp_disc, ',', fake_acc, ',', real_acc, '\n')
      
print('etth:')
display_scores(discriminative_score)
print()
# seed 12345  Final Score:  0.0731896551724138 ± 0.005795392020461213
# seed 2024  Final Score:  0.06945402298850574 ± 0.010002021083688875

# univariate Final Score:  0.004115022310361938 ± 0.003740431320553614


training: 100%|██████████| 2000/2000 [00:53<00:00, 37.50it/s]


Iter 0:  0.019809497515185015 , 0.6459138597459967 , 0.3937051352843733 



training: 100%|██████████| 2000/2000 [00:56<00:00, 35.41it/s]


Iter 1:  0.009870237437879625 , 0.40191882937603535 , 0.5783406957482055 



training: 100%|██████████| 2000/2000 [00:53<00:00, 37.48it/s]


Iter 2:  0.022742959690778553 , 0.5390668139149641 , 0.5064191054665931 



training: 100%|██████████| 2000/2000 [00:54<00:00, 36.70it/s]


Iter 3:  0.008317228050800651 , 0.4554113749309774 , 0.5279541689674213 



training: 100%|██████████| 2000/2000 [00:51<00:00, 38.91it/s]


Iter 4:  0.01656543346217554 , 0.5148398674765323 , 0.5182909994478189 

etth:
Final Score:  0.015461071231363876 ± 0.007740414368918549



## Evaluate the generated data

### 2. Predictive score

To evaluate the prediction performance on train on synthetic, test on real setting. More specifically, we use Post-hoc RNN architecture to predict one-step ahead and report the performance in terms of MAE. 

The model learns to predict the last dimension with one more step.

In [7]:
predictive_score = []
for i in range(iterations):
    temp_pred = predictive_score_metrics(ori_data, fake_data[:ori_data.shape[0]])
    predictive_score.append(temp_pred)
    print(i, ' epoch: ', temp_pred, '\n')
      
print('sine:')
display_scores(predictive_score)
print()

# univariate Final Score:  0.1605687830457955 ± 3.0256014600636085e-05

training: 100%|██████████| 5000/5000 [01:56<00:00, 42.84it/s]


0  epoch:  0.18658146358210378 



training: 100%|██████████| 5000/5000 [01:52<00:00, 44.56it/s]


1  epoch:  0.18637237935583925 



training: 100%|██████████| 5000/5000 [01:56<00:00, 42.98it/s]


2  epoch:  0.18660310973108413 



training: 100%|██████████| 5000/5000 [01:52<00:00, 44.62it/s]


3  epoch:  0.18626685993003847 



training: 100%|██████████| 5000/5000 [01:50<00:00, 45.29it/s]


4  epoch:  0.1864531908995403 

sine:
Final Score:  0.18645540069972116 ± 0.0001757676777215857

